# Abertura da RNF — Enel Distribuição São Paulo

Este notebook abre a **Renda Não Faturada (RNF)** — a energia já entregue e ainda não faturada no
fechamento do mês — por item tarifário, família e camada, com a mesma tabela de tarifas e as mesmas
famílias do P × Q do Faturamento (`SP/BW`).

**Fontes**

| Parte da RNF | Fonte |
|---|---|
| RNF calculada pelo sistema | SAP ZUBIL_146 (aba `Relatório SAP_ZUBIL_146` do relatório mensal da RNF) |
| Ajustes manuais (TOI/CNR, faturas pendentes, art. 323, iluminação pública, ajuste de carga, ajuste livre, bandeira dos ajustes) | Base de ajustes no formato novo (`BASE_ESTIMADO_RNF_POR_CLUSTER_<aaaamm>.xlsx`) |
| Alíquotas de PIS e COFINS | Mesma base de ajustes, aba `PIS_COFINS` |
| Valor de comparação | Consolidação da margem, aba `Base RNF` |
| Composição do faturado do livre | P × Q do Faturamento do mesmo mês (`SP/BW`) |

**Regras da RNF**

- A RNF contém demanda: o valor do ZUBIL inclui a receita de demanda dos dias não faturados do grupo
  A, no cativo e no livre. O bloco 1 usa a demanda não faturada que o próprio ZUBIL informa
  (coluna AW), para que essa receita seja aberta nos itens de demanda (Fio A, Fio B).
- Toda a energia é fora ponta: não há ponta nem intermediário em nenhuma classe.
- Valores monetários em milhões de reais (MR$), energia em MWh e demanda em MW.
- Reativo e bandeira não têm abertura por item: ficam em colunas à parte e voltam na conciliação.
- Primeira versão sem tarifa proporcional: vale a vigência em vigor no último dia do mês.

A lógica de valorização é a mesma do Faturamento:

| Bloco | O que é |
|---|---|
| **1 — Calculado** | quantidade não faturada × tarifa homologada, nos 37 itens e nas 4 camadas |
| **2 — Ajustado** | o bloco 1 levado ao valor registrado, linha a linha |

**O caminho:** 1. Parâmetros → 2. Ler o ZUBIL → 3. Linhas do SAP → 4. Ajustes manuais →
5. PIS/COFINS → 6. Chave tarifária e posto → 7. Tarifa da vigência → 8. Bloco 1 → 9. Bloco 2 →
10. Conferências → 11. Conciliação com a `Base RNF` → 12. Base no layout Receita Faturada →
13. Relatório detalhado → 14. Consolidado (layout Receita Não Faturada).

---
## Passo 1 — Parâmetros

Todo mês muda só `REFERENCIA`. Os nomes dos arquivos seguem os padrões abaixo, todos na pasta `src/`.
A tabela de tarifas e o P × Q do Faturamento do mesmo mês vêm da pasta do BW: a tarifa é uma fonte
única para Faturamento e RNF, e o P × Q fornece a composição do faturado do livre (Passo 9).

In [ ]:
import os
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
from pyxlsb import open_workbook

# ---------------------------------------------------------------- period and files
REFERENCIA = "2026-08"
AAAA, MM = REFERENCIA[:4], REFERENCIA[5:7]

BASE = Path(os.environ.get("RNF_BASE", r"C:\python_projects\claude\p_x_q\SP\RNF"))
ARQ_ZUBIL        = BASE / "src" / f"RNF CCS {MM}{AAAA} Fator K Nova Metodologia.xlsb"
ARQ_MANUAIS      = BASE / "src" / f"BASE_ESTIMADO_RNF_POR_CLUSTER_{AAAA}{MM}.xlsx"
ARQ_CONSOLIDACAO = BASE / "src" / f"SP_Consolidacao Margem_{AAAA}.{MM}.xlsx"
ARQ_TARIFA_LONG  = BASE.parent / "BW" / "tbl" / "tarifas" / "tarifa_long.csv"   # single tariff source
ARQ_PXQ_BW       = BASE.parent / "BW" / "1-Entrega" / "pxq" / f"pxq_{REFERENCIA}.xlsx"  # billed P x Q, same month
DIR_SAIDA        = BASE / "1-Entrega" / "rnf"
SAIDA            = DIR_SAIDA / f"rnf_{REFERENCIA}.xlsx"

# Demand of the captive group A Azul lines: the ZUBIL gives no peak / off-peak split
POSTO_DEMANDA_AZUL = "Fora ponta"

ALVO = pd.Timestamp(REFERENCIA + "-01")
FIM_MES = ALVO + pd.offsets.MonthEnd(0)
for arq in (ARQ_ZUBIL, ARQ_MANUAIS, ARQ_CONSOLIDACAO, ARQ_TARIFA_LONG, ARQ_PXQ_BW):
    assert arq.exists(), f"arquivo nao encontrado: {arq}"
print(f"referencia {REFERENCIA} -> {SAIDA}")

---
## Passo 2 — Ler o ZUBIL_146

Do relatório mensal da RNF é lida só a aba `Relatório SAP_ZUBIL_146`: a RNF calculada pelo SAP para
cada lote, classe/subclasse e categoria tarifária. As colunas são conferidas pelo cabeçalho; se o
layout mudar, o processo para.

In [ ]:
with open_workbook(str(ARQ_ZUBIL)) as livro:
    abas = {n.strip(): n for n in livro.sheets}
    with livro.get_sheet(abas["Relatório SAP_ZUBIL_146"]) as aba:
        zubil_bruto = [{c.c: c.v for c in linha if c.v not in (None, "")} for linha in aba.rows()]

# ZUBIL columns (0-based) and the header text expected in each one
COL = {"CL1": 2, "EXERC": 6, "GRP": 8, "SUB": 9, "CAT": 10,
       "R_TE": 18, "R_TUSD_CAT": 20, "R_TUSD_LIV": 22, "R_REATIVO": 26,
       "KWH_TE": 34, "KWH_TUSD_LIV": 38, "KW_CAT": 48, "R_BANDEIRA": 58}
CABECALHO = {"CAT": "Categoria Tarifa", "R_TE": "Valor Ativo Não faturado TE",
             "R_TUSD_CAT": "Valor Ativo Não fat. TUSD  TE", "R_TUSD_LIV": "Valor Ativo Não faturado TUSD",
             "R_REATIVO": "Valor Reativo Não Faturado", "KWH_TE": "Consumo Ativo Não Faturado TE"}
cab = zubil_bruto[2]
for k, txt in CABECALHO.items():
    assert str(cab.get(COL[k], "")).strip() == txt.strip(), \
        f"layout do ZUBIL mudou: coluna {COL[k]} = {cab.get(COL[k])!r}, esperado {txt!r}"

exercicios = {str(l.get(COL["EXERC"], "")).strip() for l in zubil_bruto[3:]
              if str(l.get(COL["GRP"], "")).strip() in ("ELGA", "ELGB")}
assert exercicios == {f"{AAAA}/{MM}"}, f"ZUBIL de outro periodo: {exercicios}"
print(f"{ARQ_ZUBIL.name}: exercicio {AAAA}/{MM}, {len(zubil_bruto)} linhas na aba do ZUBIL")

---
## Passo 3 — Linhas do SAP (ZUBIL_146)

As linhas do ZUBIL são somadas por classe, subclasse e categoria tarifária. Para cada uma:

- **Mercado:** dado pela categoria tarifária (categorias `LIVRE` são livre). Numa categoria livre,
  todo o valor — inclusive o que o SAP registra nas colunas de TE e "TUSD TE" — é do livre.
- **Energia (MWh):** consumo TE no cativo; consumo TUSD no livre.
- **Demanda (MW):** demanda ativa não faturada (coluna AW), no cativo e no livre do grupo A.
- **Valor (MR$, com PIS/COFINS):** ativo TE + ativo TUSD TE + ativo TUSD.
- **Não tarifáveis:** reativo e bandeira, em colunas à parte.

**Livre: energia e demanda em linhas separadas.** Nas categorias livres, o SAP registra a receita de
demanda nas colunas de TE e "TUSD TE" (sem energia correspondente) e a de energia na coluna TUSD.
Cada linha livre do ZUBIL gera, portanto, duas linhas:

| Linha | Quantidade | Valor |
|---|---|---|
| `SAP ZUBIL_146` (energia) | consumo TUSD | coluna TUSD |
| `SAP ZUBIL_146 (demanda livre)` | demanda AW | colunas de TE e TUSD TE |

Categorias e classes do SAP são traduzidas por tabelas explícitas: uma categoria nova interrompe o
processo, para ser mapeada aqui.

In [ ]:
# category -> (subgroup, modality, market)
CATEGORIA = {
    "A2_AZU": ("A2", "Azul", "CATIVO"),      "A2_LIVREAZ": ("A2", "Azul", "LIVRE"),
    "A3A_AZU": ("A3a", "Azul", "CATIVO"),    "A3A_VDE": ("A3a", "Verde", "CATIVO"),
    "A3ALIVREAZ": ("A3a", "Azul", "LIVRE"),  "A3ALIVREVD": ("A3a", "Verde", "LIVRE"),
    "A4_AZU": ("A4", "Azul", "CATIVO"),      "A4_AZURRE": ("A4", "Azul", "CATIVO"),
    "A4_VDE": ("A4", "Verde", "CATIVO"),     "A4_VDERRE": ("A4", "Verde", "CATIVO"),
    "A4_LIVREAZ": ("A4", "Azul", "LIVRE"),   "A4_LIVREVD": ("A4", "Verde", "LIVRE"),
    "AS_AZU": ("AS", "Azul", "CATIVO"),      "AS_VDE": ("AS", "Verde", "CATIVO"),
    "AS_LIVREAZ": ("AS", "Azul", "LIVRE"),   "AS_LIVREVD": ("AS", "Verde", "LIVRE"),
    "B1_RESID": ("B1", "Convencional", "CATIVO"), "B1_RESI_TB": ("B1", "Branca", "CATIVO"),
    "B2_RUR": ("B2", "Convencional", "CATIVO"),
    "B3_AESAN": ("B3", "Convencional", "CATIVO"), "B3_OUTROS": ("B3", "Convencional", "CATIVO"),
    "B3_OUTR_TB": ("B3", "Branca", "CATIVO"),
}
# SAP main class (first three letters) -> report class
CLASSE = {"REP": "RESIDENCIAL", "REB": "RESIDENCIAL", "RES": "RESIDENCIAL",
          "COM": "COMERCIAL", "IND": "INDUSTRIAL", "RUR": "RURAL", "RUA": "RURAL",
          "PPE": "PODER PÚBLICO", "PPF": "PODER PÚBLICO", "PPM": "PODER PÚBLICO",
          "SPA": "SERVIÇO PÚBLICO", "SPT": "SERVIÇO PÚBLICO"}
# SAP subclass -> social benefit (PRORET 7.1 par. 18 and 23; README do BW 3.5)
BENEFICIO = {"REBXR": "BAIXA_RENDA", "REBRBPC": "BAIXA_RENDA", "REBRIND": "BAIXA_RENDA",
             "REBRQUI": "BAIXA_RENDA", "REBXDS": "DESCONTO_SOCIAL"}

def num(v):
    try:
        f = float(v)
        return f if np.isfinite(f) else 0.0
    except (TypeError, ValueError):
        return 0.0

reg = []
for l in zubil_bruto[3:]:
    if str(l.get(COL["GRP"], "")).strip() not in ("ELGA", "ELGB"):     # subtotal rows have no group
        continue
    reg.append({"CL1": str(l.get(COL["CL1"], "")).strip(), "SUB_SAP": str(l.get(COL["SUB"], "")).strip(),
                "CATEGORIA": str(l.get(COL["CAT"], "")).strip(),
                **{k: num(l.get(c)) for k, c in COL.items()
                   if k not in ("CL1", "EXERC", "GRP", "SUB", "CAT")}})
z = pd.DataFrame(reg)

novas = sorted(set(z["CATEGORIA"]) - set(CATEGORIA))
assert not novas, f"categoria tarifaria sem mapeamento: {novas}"
novas = sorted(set(z["CL1"]) - set(CLASSE))
assert not novas, f"classe do SAP sem mapeamento: {novas}"

VALORES = ["R_TE", "R_TUSD_CAT", "R_TUSD_LIV", "R_REATIVO", "R_BANDEIRA", "KWH_TE", "KWH_TUSD_LIV", "KW_CAT"]
sap = z.groupby(["CL1", "SUB_SAP", "CATEGORIA"], as_index=False)[VALORES].sum()
sap[["SUBGRUPO", "MODALIDADE", "MERCADO"]] = pd.DataFrame(sap["CATEGORIA"].map(CATEGORIA).tolist(),
                                                          index=sap.index)
sap["CLASSE_REL"] = sap["CL1"].map(CLASSE)
sap["BENEFICIO"] = np.where(sap["MODALIDADE"] == "Convencional", sap["SUB_SAP"].map(BENEFICIO), None)
sap["ORIGEM"] = "SAP ZUBIL_146"
sap["BLOCO_BASE_RNF"] = "SISTEMA"

cativo = sap["MERCADO"] == "CATIVO"
grupo_a = sap["SUBGRUPO"].str.startswith("A")
sap["MWH"] = np.where(cativo, sap["KWH_TE"], sap["KWH_TUSD_LIV"]) / 1000.0
sap["KW"] = np.where(cativo & grupo_a, sap["KW_CAT"], 0.0)
sap["VALOR_BRUTO"] = np.where(cativo, sap["R_TE"] + sap["R_TUSD_CAT"] + sap["R_TUSD_LIV"], sap["R_TUSD_LIV"])

# free market: the demand revenue (TE and "TUSD TE" columns) becomes a line of its own, valued by AW
dem_livre = sap[~cativo & ((sap["R_TE"] + sap["R_TUSD_CAT"]) != 0)].copy()
dem_livre["ORIGEM"] = "SAP ZUBIL_146 (demanda livre)"
dem_livre["MWH"] = 0.0
dem_livre["KW"] = dem_livre["KW_CAT"]
dem_livre["VALOR_BRUTO"] = dem_livre["R_TE"] + dem_livre["R_TUSD_CAT"]
dem_livre[["R_TUSD_LIV", "KWH_TUSD_LIV", "R_REATIVO", "R_BANDEIRA"]] = 0.0
sap.loc[~cativo, ["R_TE", "R_TUSD_CAT", "KWH_TE", "KW_CAT"]] = 0.0
sap = pd.concat([sap, dem_livre], ignore_index=True)
sap = sap[(sap[["MWH", "KW", "VALOR_BRUTO", "R_REATIVO", "R_BANDEIRA"]] != 0).any(axis=1)].reset_index(drop=True)
sem_demanda = dem_livre[dem_livre["KW"] == 0]
assert sem_demanda.empty, f"valor de demanda do livre sem demanda AW: {len(sem_demanda)} linhas"

M = 1e6        # monetary values shown in millions of reais (MR$)
print(f"{len(z)} linhas do ZUBIL -> {len(sap)} linhas por origem, classe, subclasse e categoria")
r = sap.groupby(["ORIGEM", "MERCADO"]).agg(MWh=("MWH", "sum"), MW=("KW", lambda s: s.sum() / 1000),
                                           MRS=("VALOR_BRUTO", lambda s: s.sum() / M))
print(r.rename(columns={"MRS": "MR$ com PIS/COFINS"}).to_string(float_format="{:,.2f}".format))
print(f"reativo MR$ {sap['R_REATIVO'].sum() / M:,.2f} | bandeira MR$ {sap['R_BANDEIRA'].sum() / M:,.2f}")

---
## Passo 4 — Ajustes manuais

Os ajustes calculados fora do SAP vêm da base no formato novo. É usado tudo o que não é o cluster
`Sistema` (o `Sistema` é a própria RNF do SAP, já lida do ZUBIL no Passo 3).

| Cluster | Origem | Tratamento |
|---|---|---|
| `CNR` | TOI (CNR) | linha a linha, com os atributos da base |
| `PENDENTES` | Faturas pendentes | linha a linha |
| `323` | Art. 323 | linha a linha |
| `RNF IP` | Iluminação pública | linha a linha (B4a) |
| `Ajuste Carga` | Ajuste de carga | sem atributos: distribuído pelas linhas cativas do SAP |
| `Ajuste Livre` | Ajuste livre | sem atributos: distribuído pelas linhas livres do SAP |
| `bandeira Tarifária` | Bandeira dos ajustes | só bandeira: vai para os não tarifáveis |

Na distribuição, o volume segue a participação de cada linha do SAP na energia, e o valor, a
participação no valor (o mix do SAP do mesmo mercado). A base de ajustes traz o valor sem tributos e
o PIS e a COFINS em colunas próprias.

Modalidades e classes da base são traduzidas por tabelas explícitas. A modalidade `GERAÇÃO` não tem
tarifa de energia e é valorizada como Verde. A classe `RESIDENCIAL BAIXA RENDA` recebe o tratamento
de benefício social da Tarifa Social (Passo 8).

In [ ]:
COLS_MANUAIS = ["CLUSTER", "EXERCICIO", "MERCADO", "TENSAO", "SUBGRUPO", "MODALIDADE", "CLASSE_BASE",
                "LOTE", "KWH", "VALOR_LIQ", "BANDEIRA", "PIS", "COFINS", "ICMS", "TAR_LIQ", "TAR_BRUTA",
                "VALOR_BRUTO_BASE"]
CAB_MANUAIS = ["CLUSTER", "Exercício", "mercado", "Tensão", "subgrupo", "MODALIDADE", "classe_subclasse",
               "Lote", "kwh_estimado", "Valor Faturado sem PIS/COFINS", "Bandeiras Não Faturado", "PIS",
               "COFINS", "ICMS", "Tarifa Aplicada sem Imposto", "Tarifa aplicada", "VALOR_ATIVO_NAO_FATURADO"]
man = pd.read_excel(ARQ_MANUAIS, sheet_name="Base_Dados", dtype={"CLUSTER": str, "Lote": str})
for c, esperado in zip(man.columns, CAB_MANUAIS):
    assert str(c).strip().startswith(esperado), f"layout da base de ajustes mudou: {c!r}, esperado {esperado!r}"
man.columns = COLS_MANUAIS
assert set(man["EXERCICIO"].astype(str).str.strip()) == {f"{AAAA}/{MM}"}, "base de ajustes de outro periodo"
for c in ("KWH", "VALOR_LIQ", "BANDEIRA", "PIS", "COFINS", "VALOR_BRUTO_BASE"):
    man[c] = pd.to_numeric(man[c], errors="coerce").fillna(0.0)
for c in ("CLUSTER", "MERCADO", "SUBGRUPO", "MODALIDADE", "CLASSE_BASE"):
    man[c] = man[c].astype("string").str.strip()

sistema = man[man["CLUSTER"] == "Sistema"]          # used only in the check of Passo 10
man = man[man["CLUSTER"] != "Sistema"].copy()

ORIGEM_CLUSTER = {"CNR": "TOI (CNR)", "PENDENTES": "Faturas pendentes", "323": "Art. 323",
                  "RNF IP": "Iluminação pública", "Ajuste Carga": "Ajuste de carga",
                  "Ajuste Livre": "Ajuste livre", "bandeira Tarifária": "Bandeira dos ajustes"}
# block of the "Base RNF" sheet each source belongs to (check of Passo 11)
BLOCO_CLUSTER = {"CNR": "TOI", "PENDENTES": "PENDENTES", "323": "ART323", "RNF IP": "SISTEMA",
                 "Ajuste Carga": "CARGA", "Ajuste Livre": "AJUSTE_LIVRE", "bandeira Tarifária": "BANDEIRA"}
SUBGRUPO_BASE = {"A2": "A2", "A3A": "A3a", "A4": "A4", "AS": "AS", "B1": "B1", "B2": "B2", "B3": "B3", "B4": "B4"}
MODALIDADE_BASE = {"AZUL": "Azul", "VERDE": "Verde", "CONVENCIONAL": "Convencional", "BRANCA": "Branca",
                   "GERAÇÃO": "Verde"}
CLASSE_BASE = {"01 - RESIDENCIAL": "RESIDENCIAL", "01b - RESIDENCIAL BAIXA RENDA": "RESIDENCIAL",
               "02 - INDUSTRIAL": "INDUSTRIAL", "03 - COMERCIAL": "COMERCIAL", "04 - RURAL": "RURAL",
               "05 - PODERES PÚBLICOS": "PODER PÚBLICO", "06 - ILUMINAÇÃO PÚBLICA": "ILUMINAÇÃO PÚBLICA",
               "07 - SERVIÇOS PÚBLICOS": "SERVIÇO PÚBLICO", "08 - CONSUMO PRÓPRIO": "CONSUMO PRÓPRIO"}

novos = sorted(set(man["CLUSTER"]) - set(ORIGEM_CLUSTER))
assert not novos, f"cluster sem mapeamento: {novos}"
com_attr = man["SUBGRUPO"].notna()
for col, mapa in (("SUBGRUPO", SUBGRUPO_BASE), ("MODALIDADE", MODALIDADE_BASE), ("CLASSE_BASE", CLASSE_BASE)):
    novos = sorted(set(man.loc[com_attr, col].dropna()) - set(mapa))
    assert not novos, f"{col} sem mapeamento: {novos}"

man["ORIGEM"] = man["CLUSTER"].map(ORIGEM_CLUSTER)
man["BLOCO_BASE_RNF"] = man["CLUSTER"].map(BLOCO_CLUSTER)
man["PIS_COFINS"] = man["PIS"] + man["COFINS"]
man["VALOR_BRUTO"] = man["VALOR_LIQ"] + man["PIS_COFINS"]
man["MWH"] = man["KWH"] / 1000.0

# bandeira of every manual row goes to the non-tariff items
nt_man = man.copy()
nt_man["CLASSE_REL"] = nt_man["CLASSE_BASE"].map(CLASSE_BASE).fillna("AJUSTES SEM CLASSE")
BANDEIRA_AJUSTES = nt_man.groupby(["MERCADO", "CLASSE_REL"])["BANDEIRA"].sum()

COLS_LINHA = ["ORIGEM", "BLOCO_BASE_RNF", "MERCADO", "CLASSE_REL", "CL1", "SUB_SAP", "CATEGORIA",
              "SUBGRUPO", "MODALIDADE", "BENEFICIO", "MWH", "KW", "VALOR_BRUTO", "PIS_COFINS", "VALOR_LIQ"]
linhas_aj = []
# 1) rows with tariff attributes, taken one by one
d = man[com_attr & ((man["MWH"] != 0) | (man["VALOR_BRUTO"] != 0))].copy()
d["SUBGRUPO"] = d["SUBGRUPO"].map(SUBGRUPO_BASE)
d["MODALIDADE"] = d["MODALIDADE"].map(MODALIDADE_BASE)
d["CLASSE_REL"] = d["CLASSE_BASE"].map(CLASSE_BASE)
d["BENEFICIO"] = np.where(d["CLASSE_BASE"].str.contains("BAIXA RENDA") & (d["MODALIDADE"] == "Convencional"),
                          "BAIXA_RENDA", None)
d["CL1"], d["SUB_SAP"] = d["CLASSE_BASE"], ""
d["CATEGORIA"] = d["SUBGRUPO"] + " " + d["MODALIDADE"] + " (base de ajustes)"
d["KW"] = 0.0
linhas_aj.append(d[COLS_LINHA])

# 2) rows without attributes, spread over the SAP lines of the same market (energy and value mix)
sap["PIS_COFINS"], sap["VALOR_LIQ"] = np.nan, np.nan      # filled in Passo 5
for _, r in man[~com_attr & ((man["MWH"] != 0) | (man["VALOR_BRUTO"] != 0))].iterrows():
    ref = sap[(sap["MERCADO"] == r["MERCADO"]) & (sap["MWH"] > 0) & (sap["VALOR_BRUTO"] > 0)]
    assert not ref.empty, f"sem linhas do SAP para distribuir {r['CLUSTER']}"
    f_e = ref["MWH"] / ref["MWH"].sum()
    f_v = ref["VALOR_BRUTO"] / ref["VALOR_BRUTO"].sum()
    novo = ref[["CL1", "SUB_SAP", "CATEGORIA", "SUBGRUPO", "MODALIDADE", "MERCADO", "CLASSE_REL",
                "BENEFICIO"]].copy()
    novo["ORIGEM"], novo["BLOCO_BASE_RNF"] = r["ORIGEM"], r["BLOCO_BASE_RNF"]
    novo["MWH"] = r["MWH"] * f_e
    novo["KW"] = ref["KW"] * (novo["MWH"] / ref["MWH"])          # demand follows the energy share
    novo["VALOR_BRUTO"] = r["VALOR_BRUTO"] * f_v
    novo["PIS_COFINS"] = r["PIS_COFINS"] * f_v
    novo["VALOR_LIQ"] = r["VALOR_LIQ"] * f_v
    linhas_aj.append(novo[COLS_LINHA])

ajustes = pd.concat(linhas_aj, ignore_index=True)
r = man.groupby(["ORIGEM", "MERCADO"])[["MWH", "VALOR_LIQ", "PIS_COFINS", "BANDEIRA"]].sum()
r[["VALOR_LIQ", "PIS_COFINS", "BANDEIRA"]] /= M
print("MWh e MR$")
print(r.to_string(float_format="{:,.2f}".format))
d = (ajustes["VALOR_BRUTO"].sum() - man["VALOR_BRUTO"].sum(), ajustes["MWH"].sum() - man["MWH"].sum())
assert abs(d[0]) < 0.01 and abs(d[1]) < 1e-6, f"ajustes perderam valor ou volume na distribuicao: {d}"
print(f"{len(man)} linhas de ajuste -> {len(ajustes)} linhas | bandeira dos ajustes MR$ {BANDEIRA_AJUSTES.sum() / M:,.2f}")

---
## Passo 5 — PIS/COFINS das linhas do SAP

Os valores do ZUBIL contêm PIS/COFINS; a tarifa homologada não tem tributos, então o alvo do bloco 2
é o valor **líquido**. É usada a mesma convenção da base de ajustes, com as alíquotas da aba
`PIS_COFINS` dessa base:

`valor líquido = valor com tributos ÷ (1 + alíquota de PIS + alíquota de COFINS)`

O tributo retirado vai para os não tarifáveis e volta na conciliação.

In [ ]:
pc = pd.read_excel(ARQ_MANUAIS, sheet_name="PIS_COFINS", header=None)
assert str(pc.iat[0, 3]).strip() == "PIS" and str(pc.iat[0, 4]).strip() == "COFINS", "layout da aba PIS_COFINS mudou"
ALIQ_PIS, ALIQ_COFINS = float(pc.iat[1, 3]), float(pc.iat[1, 4])
ALIQUOTA = ALIQ_PIS + ALIQ_COFINS
print(f"aliquotas: PIS {ALIQ_PIS:.4%} + COFINS {ALIQ_COFINS:.4%} = {ALIQUOTA:.4%}")

sap["VALOR_LIQ"] = sap["VALOR_BRUTO"] / (1 + ALIQUOTA)
sap["PIS_COFINS"] = sap["VALOR_BRUTO"] - sap["VALOR_LIQ"]
print(f"SAP: valor com tributos MR$ {sap['VALOR_BRUTO'].sum() / M:,.2f} -> liquido MR$ {sap['VALOR_LIQ'].sum() / M:,.2f}")

---
## Passo 6 — Chave tarifária e posto horário

| Origem | Vai para a chave como |
|---|---|
| Grupo A e B3 | classe e subclasse `Não se aplica` (a tarifa não varia por classe) |
| B1 | classe `Residencial`; subclasse `Residencial`, ou `Baixa Renda` / Desconto Social pelo benefício (Passo 8) |
| B2 / B4 | classe `Rural` / `Iluminação pública`; subclasse `Não se aplica` / `Iluminação pública – B4a` |
| Detalhe e acessante | `Não se aplica` (APE, SCEE e fonte incentivada não são identificados: o efeito deles fica no bloco 2) |

**Posto horário:** toda a energia é fora ponta. No grupo A (Azul e Verde) e na Branca, a energia vai
para a tarifa `Fora ponta`; no Convencional, não há posto. A demanda do Azul vai para o posto de
`POSTO_DEMANDA_AZUL` (Passo 1); a demanda do Verde não tem posto.

In [ ]:
NSA = "Não se aplica"
linhas = pd.concat([sap[COLS_LINHA], ajustes], ignore_index=True)
linhas["CLASSE"] = linhas["SUBGRUPO"].map({"B1": "Residencial", "B2": "Rural",
                                           "B4": "Iluminação pública"}).fillna(NSA)
linhas["SUBCLASSE"] = np.select([linhas["SUBGRUPO"] == "B1", linhas["SUBGRUPO"] == "B4"],
                                ["Residencial", "Iluminação pública – B4a"], NSA)
linhas["DETALHE"], linhas["ACESSANTE"] = NSA, NSA

fp_energia = linhas["SUBGRUPO"].str.startswith("A") | (linhas["MODALIDADE"] == "Branca")
linhas["POSTO_E"] = np.where(fp_energia, "Fora ponta", NSA)
linhas["POSTO_D"] = np.where(linhas["MODALIDADE"] == "Azul", POSTO_DEMANDA_AZUL, NSA)
r = linhas.groupby(["ORIGEM", "MERCADO"]).agg(MWh=("MWH", "sum"), MW=("KW", lambda s: s.sum() / 1000),
                                              MRS_liquido=("VALOR_LIQ", lambda s: s.sum() / M))
print(r.to_string(float_format="{:,.2f}".format))

---
## Passo 7 — Tarifa da vigência

Vale a vigência em vigor no último dia do mês de referência. No mês do reajuste (4 de julho, em SP)
há duas vigências no mês; a tarifa proporcional ainda não está implementada nesta versão, e o
processo avisa. As quatro camadas vêm da mesma tabela (TA = BE + CVA + FIN em cada item).

In [ ]:
CAMADAS = ["TA", "BE", "CVA", "FIN"]
ATRIBUTOS = ["SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE", "ACESSANTE", "POSTO", "UNIDADE"]

tl = pd.read_csv(ARQ_TARIFA_LONG, sep=";", decimal=",", encoding="utf-8-sig")
tl["VIGENCIA_INICIO"] = pd.to_datetime(tl["VIGENCIA_INICIO"])
tl["VIGENCIA_FIM"] = pd.to_datetime(tl["VIGENCIA_FIM"])
COMPONENTES = [c for c in tl.columns if c not in {"ANO_VIGENCIA", "VIGENCIA_INICIO", "VIGENCIA_FIM",
               "CAMADA", "ORIGEM", "Total TUSD", "Total TE", *ATRIBUTOS}]

vig = tl.groupby("ANO_VIGENCIA")[["VIGENCIA_INICIO", "VIGENCIA_FIM"]].first()
no_mes = vig[(vig["VIGENCIA_INICIO"] <= FIM_MES) & (vig["VIGENCIA_FIM"] >= ALVO)]
no_fim = vig[(vig["VIGENCIA_INICIO"] <= FIM_MES) & (vig["VIGENCIA_FIM"] >= FIM_MES)]
assert len(no_fim) == 1, f"nenhuma vigencia (ou mais de uma) cobre {FIM_MES.date()}"
ANO_VIGENCIA = int(no_fim.index[0])
if len(no_mes) > 1:
    print(f"ATENCAO: mes de reajuste ({list(no_mes.index)}); tarifa proporcional nao implementada, "
          f"usada a vigencia {ANO_VIGENCIA}")
_t = tl[tl["ANO_VIGENCIA"] == ANO_VIGENCIA]
TARIFA = {c: _t[_t["CAMADA"] == c].set_index(ATRIBUTOS)[COMPONENTES].astype(float) for c in CAMADAS}
print(f"vigencia {ANO_VIGENCIA}: {no_fim.iloc[0, 0].date()} a {no_fim.iloc[0, 1].date()}")

def tarifa(attrs, posto, unidade, subclasse=None):
    """Tariff per layer for one key; falls back to subclass 'Nao se aplica'."""
    base = {**attrs, "POSTO": posto, "UNIDADE": unidade}
    if subclasse:
        base["SUBCLASSE"] = subclasse
    k = tuple(base[a] for a in ATRIBUTOS)
    if k not in TARIFA["TA"].index:
        k = tuple({**base, "SUBCLASSE": NSA}[a] for a in ATRIBUTOS)
    if k not in TARIFA["TA"].index:
        raise KeyError(f"sem tarifa: {tuple(base[a] for a in ATRIBUTOS)}")
    return {c: TARIFA[c].loc[k].values for c in CAMADAS}

---
## Passo 8 — Bloco 1: o valor pela tarifa

- **Parcelas:** cativo paga TUSD e TE na energia; livre, só TUSD. A demanda é só TUSD.
- **Benefícios sociais:** a faixa de consumo não é informada. Como no Faturamento (procedimento do
  BW, Passo 8), cada linha de benefício é valorizada nas duas faixas, e a participação da faixa 02 é a
  que reproduz o valor registrado: `s2 = (R − A) / (B − A)`, limitada a [0, 1].
  - Tarifa Social: faixa 01 = gratuidade (A = 0); faixa 02 = tarifa `Baixa Renda`.
  - Desconto Social: faixa 01 e faixa 02 pelas chaves próprias.

In [ ]:
ATTR_FIXOS = ["SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE", "ACESSANTE"]
COLS_TE = np.array([c.startswith("TE") for c in COMPONENTES])

UNIDADES = ("E", "D")                 # energy (MWh) and demand (MW), as in the Receita Faturada layout

def valor_partes(r, subclasse=None):
    """Block-1 value per unit and layer for one line: {"E": {layer: vector}, "D": {...}}."""
    attrs = {a: r[a] for a in ATTR_FIXOS}
    e = tarifa(attrs, r["POSTO_E"], "MWh", subclasse)
    out = {"E": {}, "D": {}}
    for c in CAMADAS:
        v = e[c] * r["MWH"]
        out["E"][c] = v if r["MERCADO"] == "CATIVO" else np.where(COLS_TE, 0.0, v)
        out["D"][c] = np.zeros(len(COMPONENTES))
    if r["KW"]:
        d = tarifa({**attrs, "CLASSE": NSA, "SUBCLASSE": NSA}, r["POSTO_D"], "kW")
        for c in CAMADAS:
            out["D"][c] = np.where(COLS_TE, 0.0, d[c] * r["KW"])
    return out

B1U = {u: {c: np.zeros((len(linhas), len(COMPONENTES))) for c in CAMADAS} for u in UNIDADES}
linhas["FAIXA2_PCT"] = np.nan
for i, r in linhas.iterrows():
    ben = r["BENEFICIO"]
    if ben == "BAIXA_RENDA":
        B = valor_partes(r, "Baixa Renda")
        A = {u: {c: np.zeros(len(COMPONENTES)) for c in CAMADAS} for u in UNIDADES}
    elif ben == "DESCONTO_SOCIAL":
        A = valor_partes(r, "Residencial Desconto Social – faixa 01")
        B = valor_partes(r, "Residencial Desconto Social – faixa 02")
    else:
        v = valor_partes(r)
        for u in UNIDADES:
            for c in CAMADAS:
                B1U[u][c][i] = v[u][c]
        continue
    a = sum(A[u]["TA"].sum() for u in UNIDADES)
    b = sum(B[u]["TA"].sum() for u in UNIDADES)
    R = r["VALOR_LIQ"]
    s2 = float(np.clip((R - a) / (b - a), 0.0, 1.0)) if b != a else 1.0
    linhas.at[i, "FAIXA2_PCT"] = s2
    for u in UNIDADES:
        for c in CAMADAS:
            B1U[u][c][i] = (1 - s2) * A[u][c] + s2 * B[u][c]

B1U = {u: {c: pd.DataFrame(B1U[u][c], index=linhas.index, columns=COMPONENTES) for c in CAMADAS}
       for u in UNIDADES}
B1 = {c: B1U["E"][c] + B1U["D"][c] for c in CAMADAS}
linhas["BLOCO1_TA"] = B1["TA"].sum(axis=1)
linhas["DIFERENCA"] = linhas["VALOR_LIQ"] - linhas["BLOCO1_TA"]
g = linhas.groupby("MERCADO")[["VALOR_LIQ", "BLOCO1_TA", "DIFERENCA"]].sum() / M
g["DIF_%"] = g["DIFERENCA"] / g["VALOR_LIQ"] * 100
print("MR$, liquido de PIS/COFINS")
print(g.to_string(float_format="{:,.2f}".format))

---
## Passo 9 — Bloco 2: levar ao valor registrado

A mesma regra do Faturamento (README do BW, seção 4):

- **Livre do grupo A, linha de demanda — regime fio.** O bloco 1 é a demanda AW × tarifa de demanda
  de TUSD, sem desconto. A diferença para o valor registrado é a **aplicação do desconto** de fonte
  incentivada e APE, que incide sobre o fio (Fio A e Fio B, na composição da tarifa de demanda) — o
  mesmo tratamento do Faturamento, estimado pelo valor da receita.
- **Grupo A, linhas de energia — composição do faturado.** O ZUBIL traz a energia sem posto horário
  e, no livre, só parte da demanda; o valor registrado, porém, contém a energia de ponta e a demanda,
  já com os descontos. A diferença para o bloco 1 é distribuída na composição do **faturado do mesmo
  mês** (bloco 2 do P × Q do BW), mesmo mercado, subgrupo e modalidade, só nos **postos que o bloco 1
  não valoriza**, item a item e camada a camada:

  | Mercado | O bloco 1 valoriza | A diferença segue o faturado de |
  |---|---|---|
  | Livre | energia fora ponta | energia de ponta + demanda (ponta e fora ponta) |
  | Cativo | energia fora ponta + demanda AW (fora ponta) | energia de ponta + demanda de ponta (Azul) |

  Sem faturado no grupo, vale o regime fio no livre e o rateio proporcional no cativo.
- **Demais linhas — rateio proporcional.** A diferença é distribuída por todos os itens e camadas na
  proporção do bloco 1, como na 2ª passada do Faturamento.

Linha com bloco 1 zero e valor registrado não tem o que escalar: fica como resíduo, informado à parte.

In [ ]:
FAMILIA = {
    "FIO A":   ["TUSD_RB", "TUSD_FR", "TUSD_CCT", "TUSD_CCD", "TUSD_CUSD", "TUSDG_T",
                "TE_TRANSPORTE_ITAIPU", "TE_TUST_ITAIPU", "TE_TUST_CI"],
    "FIO B":   ["TUSD_FioB"],
    "ENCARGOS":["TUSD_CDE_COVID", "TUSD_TFSEE", "TUSD_PeD", "TUSD_ONS", "TUSD_CCC", "TUSD_CDE",
                "TUSD_PROINFA", "Liminar1", "TUSDG_ONS", "TE_CDE_COVID", "TE_CDE_ELET",
                "TE_CDE_GD", "TE_PeD", "TE_ESSERR", "TE_CFURH"],
    "PERDAS":  ["TUSD_PT", "TUSD_Per_RB_D", "TUSD_PNT", "TE_Per_RB"],
    "ENERGIA": ["TE_ENERGIA", "TE_ANGRA"],
    "RI":      ["TUSD_RI"],
    "OUTROS":  ["TUSD Subsidio", "TUSD BENEFICIO_L14299", "TUSD Outros",
                "TE Subsidio", "TE BENEFICIO_L14299"],
}
assert not [c for c in COMPONENTES if c not in {x for v in FAMILIA.values() for x in v}]
FIO = np.array([c in FAMILIA["FIO A"] + FAMILIA["FIO B"] for c in COMPONENTES])

# billed composition of the month (BW block 2), only in the time-of-use slots that block 1 does not
# value: free market -> peak energy + demand; captive -> peak energy + peak demand (Azul)
POSTOS_FAT = {"LIVRE": {"EP", "DP", "DFP", "D"}, "CATIVO": {"EP", "DP"}}
MERCADO_BW = {"Cliente Livre": "LIVRE", "Cliente Híbrido": "LIVRE", "Cliente Cativo": "CATIVO"}
bw_ta = pd.read_excel(ARQ_PXQ_BW, sheet_name="Componentes")
bw_be = pd.read_excel(ARQ_PXQ_BW, sheet_name="Componentes_BE")
assert len(bw_ta) == len(bw_be) and (bw_ta["Chave"].astype(str) == bw_be["Chave"].astype(str)).all(), \
    "abas Componentes e Componentes_BE do P x Q nao estao alinhadas"
itens_bw = {c[4:] for c in bw_ta.columns if c.startswith("[2] ") and not c.startswith("[2] Total")}
assert not [c for c in COMPONENTES if c not in itens_bw], "item tarifario ausente no P x Q do BW"
cab_bw = bw_ta["Chave"].astype(str).str.extract(r"^(A2|A3a|A4|AS)(Azul|Verde)")
posto_bw = bw_ta["Chave"].astype(str).str.extract(r"(EFP|EP|DFP|DP|D)(?:MWh|kW)$")[0]
merc_bw = bw_ta["Flag Cli.Liv./ Cat."].map(MERCADO_BW)
no_posto = pd.Series([m in POSTOS_FAT and p in POSTOS_FAT[m] for m, p in zip(merc_bw, posto_bw)], index=bw_ta.index)
sel = (bw_ta["Tarifável?"] == "SIM") & cab_bw[0].notna() & no_posto
ta = bw_ta.loc[sel, [f"[2] {c}" for c in COMPONENTES]].to_numpy(float)
be = bw_be.loc[sel, [f"[2] BE · {c}" if f"[2] BE · {c}" in bw_be.columns else f"[2] {c}" for c in COMPONENTES]].to_numpy(float)
cva_t = bw_ta.loc[sel, "[2] Total CVA"].to_numpy(float)
fin_t = bw_ta.loc[sel, "[2] Total FIN"].to_numpy(float)
w_cva = np.divide(cva_t, cva_t + fin_t, out=np.full_like(cva_t, 0.5), where=(cva_t + fin_t) != 0)[:, None]
camadas_bw = {"TA": ta, "BE": be, "CVA": (ta - be) * w_cva, "FIN": (ta - be) * (1 - w_cva)}
grupo_bw = list(zip(merc_bw[sel], cab_bw.loc[sel, 0], cab_bw.loc[sel, 1]))
unid_bw = np.where(posto_bw[sel].str.startswith("E"), "E", "D")
COMP_FAT, COMP_FAT_U = {}, {}
for g in set(grupo_bw):
    m = np.array([x == g for x in grupo_bw])
    COMP_FAT[g] = {c: camadas_bw[c][m].sum(axis=0) for c in CAMADAS}
    COMP_FAT_U[g] = {u: {c: camadas_bw[c][m & (unid_bw == u)].sum(axis=0) for c in CAMADAS} for u in UNIDADES}
print("composicao do faturado nos postos que o bloco 1 nao valoriza (MR$, bloco 2 do BW):")
print(pd.Series({" ".join(g): v["TA"].sum() / M for g, v in sorted(COMP_FAT.items())})
      .to_string(float_format="{:,.2f}".format))

B2U = {u: {c: B1U[u][c].copy() for c in CAMADAS} for u in UNIDADES}
grupo_a = linhas["SUBGRUPO"].str.startswith("A")
livre_a = (linhas["MERCADO"] == "LIVRE") & grupo_a
tem_fat = pd.Series([(m, sg, mo) in COMP_FAT and COMP_FAT[(m, sg, mo)]["TA"].sum() != 0
                     for m, sg, mo in zip(linhas["MERCADO"], linhas["SUBGRUPO"], linhas["MODALIDADE"])],
                    index=linhas.index)
linhas["REGIME"] = np.select(
    [livre_a & (linhas["ORIGEM"] == "SAP ZUBIL_146 (demanda livre)"), grupo_a & tem_fat, livre_a],
    ["fio", "faturado", "fio"], "proporcional")
linhas["AJUSTE_FIO"], linhas["AJUSTE_FATURADO"], linhas["FATOR"], linhas["RESIDUO"] = 0.0, 0.0, 1.0, 0.0
for i, r in linhas.iterrows():
    dif = r["DIFERENCA"]
    if r["REGIME"] == "faturado":
        g = (r["MERCADO"], r["SUBGRUPO"], r["MODALIDADE"])
        base = COMP_FAT[g]["TA"].sum()
        for u in UNIDADES:
            for c in CAMADAS:
                B2U[u][c].loc[i] = B1U[u][c].loc[i].values + dif * COMP_FAT_U[g][u][c] / base
        linhas.at[i, "AJUSTE_FATURADO"] = dif
    elif r["REGIME"] == "fio":
        # the wires adjustment follows the demand tariff: it is a demand (D) amount
        d = tarifa({"SUBGRUPO": r["SUBGRUPO"], "MODALIDADE": r["MODALIDADE"], "CLASSE": NSA,
                    "SUBCLASSE": NSA, "DETALHE": NSA, "ACESSANTE": NSA}, r["POSTO_D"], "kW")
        base = (d["TA"] * FIO).sum()
        for c in CAMADAS:
            B2U["D"][c].loc[i] = B1U["D"][c].loc[i].values + dif * (d[c] * FIO) / base
        linhas.at[i, "AJUSTE_FIO"] = dif
    elif r["BLOCO1_TA"]:
        f = r["VALOR_LIQ"] / r["BLOCO1_TA"]
        for u in UNIDADES:
            for c in CAMADAS:
                B2U[u][c].loc[i] = B1U[u][c].loc[i] * f
        linhas.at[i, "FATOR"] = f
    else:
        linhas.at[i, "RESIDUO"] = r["VALOR_LIQ"]
B2 = {c: B2U["E"][c] + B2U["D"][c] for c in CAMADAS}
linhas["BLOCO2_TA"] = B2["TA"].sum(axis=1)
# on the free-market demand lines the difference is the discount (incentivized source / APE) on the wires
linhas["DESCONTO"] = np.where(linhas["ORIGEM"] == "SAP ZUBIL_146 (demanda livre)", linhas["AJUSTE_FIO"], 0.0)
linhas.loc[linhas["DESCONTO"] != 0, "AJUSTE_FIO"] = 0.0
r = linhas.groupby(["REGIME", "ORIGEM"])[["VALOR_LIQ", "BLOCO1_TA", "DESCONTO", "AJUSTE_FIO", "AJUSTE_FATURADO",
                                          "BLOCO2_TA", "RESIDUO"]].sum() / M
print("MR$, liquido de PIS/COFINS")
print(r.to_string(float_format="{:,.2f}".format))
d = linhas[linhas["ORIGEM"] == "SAP ZUBIL_146 (demanda livre)"].groupby(["SUBGRUPO", "MODALIDADE"])[
    ["KW", "VALOR_LIQ", "BLOCO1_TA"]].sum()
d["desconto %"] = (1 - d["VALOR_LIQ"] / d["BLOCO1_TA"]) * 100
d["MW"] = d.pop("KW") / 1000
d[["VALOR_LIQ", "BLOCO1_TA"]] /= M
print("\ndemanda livre: desconto estimado sobre o valor sem desconto")
print(d[["MW", "VALOR_LIQ", "BLOCO1_TA", "desconto %"]].to_string(float_format="{:,.2f}".format))

---
## Passo 10 — Conferências

1. Bloco 2 + resíduo = valor líquido, linha a linha.
2. TA = BE + CVA + FIN no bloco 2.
3. Famílias = componentes.
4. ZUBIL × cluster `Sistema` da base de ajustes: energia, valor com tributos e valor líquido por
   mercado. A base de ajustes também traz o `Sistema`, calculado a partir do ZUBIL; os dois devem ser
   iguais.
5. Medida, não aprovação: fator do rateio proporcional por subgrupo (quanto a tarifa explica sozinha).

In [ ]:
ok = True
d = (linhas["BLOCO2_TA"] + linhas["RESIDUO"] - linhas["VALOR_LIQ"]).abs().max()
print(f"1. bloco 2 + residuo x valor liquido ..... diferenca maxima {d:,.6f}"); ok &= d < 0.01
d = (B2["TA"] - B2["BE"] - B2["CVA"] - B2["FIN"]).abs().max().max()
print(f"2. TA = BE + CVA + FIN ................... diferenca maxima {d:,.8f}"); ok &= d < 0.01
d = max(abs(B2[c].sum().sum() - sum(B2[c][l].sum().sum() for l in FAMILIA.values())) for c in CAMADAS)
print(f"3. familias x componentes ................ diferenca maxima {d:,.8f}"); ok &= d < 0.01

print("4. ZUBIL x cluster Sistema da base de ajustes:")
zb = sap.groupby("MERCADO").agg(KWH=("MWH", lambda s: s.sum() * 1000), BRUTO=("VALOR_BRUTO", "sum"),
                                LIQ=("VALOR_LIQ", "sum"))
sb = sistema.groupby("MERCADO").agg(KWH=("KWH", "sum"), BRUTO=("VALOR_BRUTO_BASE", "sum"), LIQ=("VALOR_LIQ", "sum"))
cmp4 = zb.join(sb, lsuffix="_ZUBIL", rsuffix="_BASE")
for c in ("KWH", "BRUTO", "LIQ"):
    cmp4[f"DIF_{c}"] = cmp4[f"{c}_ZUBIL"] - cmp4[f"{c}_BASE"]
ok &= bool((cmp4[["DIF_BRUTO", "DIF_LIQ"]].abs() < 1.0).all().all())
cmp4_mostra = pd.DataFrame({"dif MWh": cmp4["DIF_KWH"] / 1000, "dif MR$ com tributos": cmp4["DIF_BRUTO"] / M,
                            "dif MR$ liquido": cmp4["DIF_LIQ"] / M})
print(cmp4_mostra.to_string(float_format="{:,.4f}".format))
print(f"   residuo nao ajustavel ................ MR$ {linhas['RESIDUO'].sum() / M:,.2f}")
prop = linhas[linhas["REGIME"] == "proporcional"].groupby(["SUBGRUPO", "MODALIDADE"])[["VALOR_LIQ", "BLOCO1_TA"]].sum()
print("\n5. valor liquido / bloco 1 no rateio proporcional:")
print((prop["VALOR_LIQ"] / prop["BLOCO1_TA"]).map("{:.3f}".format).to_string())
print("\nTODAS AS CONFERENCIAS PASSARAM" if ok else "\nALGUMA CONFERENCIA FALHOU - NAO USAR")

---
## Passo 11 — Conciliação com a `Base RNF`

O total aberto mais os não tarifáveis (PIS/COFINS, reativo e bandeira) é comparado com a aba
`Base RNF` da consolidação da margem, coluna "RNF R$" do mês, bloco a bloco.

A `Base RNF` agrupa os valores de forma diferente da abertura; para comparar, a abertura é remontada
no mesmo agrupamento:

| Bloco da `Base RNF` | Composição na abertura (valores com tributos) |
|---|---|
| Consumo Cativo | colunas de TE e TUSD TE do ZUBIL (todas as categorias) + reativo do ZUBIL + iluminação pública |
| TOI | cluster `CNR` (cativo e livre) |
| Pendentes | cluster `PENDENTES` |
| Artigo 113 | cluster `323` |
| Carga | cluster `Ajuste Carga` |
| Bandeira | bandeira do ZUBIL + bandeira dos ajustes |
| Total Livres | coluna TUSD do ZUBIL + cluster `Ajuste Livre` |

In [ ]:
import datetime

ws = openpyxl.load_workbook(ARQ_CONSOLIDACAO, read_only=True, data_only=True)["Base RNF"]
grade = [list(r) for r in ws.iter_rows(values_only=True)]
# month columns: the date of row 2 marks the first of five columns (kWh, tariffs, PIS/COFINS, RNF R$)
col_mes = next(j for j, v in enumerate(grade[1])
               if isinstance(v, datetime.datetime) and pd.Timestamp(v) == ALVO)
COL_RS = col_mes + 4
# block totals are in the summary section that starts at the "Consumo Cativo" row
INICIO_RESUMO = next(i for i, r in enumerate(grade) if str(r[1] or "").strip() == "Consumo Cativo")

def valor_base_rnf(rotulo, col_rotulo):
    for r in grade[INICIO_RESUMO:]:
        if str(r[col_rotulo] or "").strip() == rotulo:
            return num(r[COL_RS])
    raise KeyError(f"rotulo nao encontrado na Base RNF: {rotulo}")

ref = pd.Series({"Consumo Cativo": valor_base_rnf("Consumo Cativo", 1), "TOI": valor_base_rnf("TOI", 1),
                 "Pendentes": valor_base_rnf("Pendentes", 1), "Artigo 113": valor_base_rnf("Artigo 113", 1),
                 "Carga": valor_base_rnf("Carga", 1), "Bandeira": valor_base_rnf("Bandeira", 1),
                 "Total Livres": valor_base_rnf("Total Livres", 0)})
TOTAL_BASE_RNF = valor_base_rnf("Total RNF", 0)
assert abs(ref.sum() - TOTAL_BASE_RNF) < 1.0, "blocos da Base RNF nao somam o total"

bruto = linhas.groupby("BLOCO_BASE_RNF")["VALOR_BRUTO"].sum()
REATIVO = sap["R_REATIVO"].sum()
BANDEIRA = sap["R_BANDEIRA"].sum() + BANDEIRA_AJUSTES.sum()
ip = linhas.loc[linhas["ORIGEM"] == "Iluminação pública", "VALOR_BRUTO"].sum()
aberto = pd.Series({
    "Consumo Cativo": sap["R_TE"].sum() + sap["R_TUSD_CAT"].sum() + REATIVO + ip,
    "TOI": bruto.get("TOI", 0.0), "Pendentes": bruto.get("PENDENTES", 0.0),
    "Artigo 113": bruto.get("ART323", 0.0), "Carga": bruto.get("CARGA", 0.0), "Bandeira": BANDEIRA,
    "Total Livres": sap["R_TUSD_LIV"].sum() + bruto.get("AJUSTE_LIVRE", 0.0)})
CONC_BASE_RNF = pd.DataFrame({"Base RNF (R$)": ref, "Abertura (R$)": aberto})
CONC_BASE_RNF.loc["Total"] = CONC_BASE_RNF.sum()
CONC_BASE_RNF["Diferença (R$)"] = CONC_BASE_RNF["Abertura (R$)"] - CONC_BASE_RNF["Base RNF (R$)"]

total_aberto = pd.Series({"Valor aberto por item (liquido)": linhas["BLOCO2_TA"].sum() + linhas["RESIDUO"].sum(),
                          "PIS/COFINS": linhas["PIS_COFINS"].sum(), "Reativo": REATIVO, "Bandeira": BANDEIRA})
total_aberto["Total"] = total_aberto.sum()
CONCILIACAO = pd.DataFrame({"R$": total_aberto})
CONCILIACAO.loc["Base RNF (total do mes)"] = TOTAL_BASE_RNF
CONCILIACAO.loc["Diferença x Base RNF"] = total_aberto["Total"] - TOTAL_BASE_RNF
assert abs(total_aberto["Total"] - CONC_BASE_RNF.loc["Total", "Abertura (R$)"]) < 1.0, \
    "o total aberto difere da soma dos blocos"

CONCILIACAO = CONCILIACAO / M
CONCILIACAO.columns = ["MR$"]
CONC_BASE_RNF = CONC_BASE_RNF / M
CONC_BASE_RNF.columns = ["Base RNF (MR$)", "Abertura (MR$)", "Diferença (MR$)"]
print(CONCILIACAO["MR$"].map("{:,.2f}".format).to_string())
print()
print(CONC_BASE_RNF.to_string(float_format="{:,.2f}".format))
if abs(CONCILIACAO.loc["Diferença x Base RNF", "MR$"]) >= 1.0 / M:
    print("\nATENCAO: diferenca com a Base RNF; ver os blocos acima")

---
## Passo 12 — Base no layout Receita Faturada

O consolidado segue o layout **Receita Faturada** da área (modelo `BW/tbl/modelo/De-Para_Modelo_PyC.xlsx`,
o mesmo do Faturamento), com o mesmo conversor de itens para grupos do layout (aba "Instruções").

| Campo | Regra na RNF |
|---|---|
| `TipoFaturamento` | `RNF` (SAP ZUBIL_146, energia e demanda); `RNF_TOI`, `RNF_PENDENTES`, `RNF_ART323`, `RNF_CARGA`, `RNF_IP`, `RNF_AJUSTE_LIVRE` (ajustes manuais); `RNF_BANDEIRA` (bandeira dos ajustes) |
| `OpcaoCompra` | CATIVO ou LIVRE |
| `Classe` | classe do layout; Tarifa Social e Desconto Social em `Residencial Baixa Renda` |
| `Unidade` | E (energia, MWh) e D (demanda, MW), separadas desde o bloco 1 |
| Parcela B a TFSEE | bloco 2 na camada BE, item a item, pelo de-para do modelo |
| Base Financeira / Base CVA | totais FIN e CVA do bloco 2 |
| Bandeira | bandeira do ZUBIL e dos ajustes |
| UD/ER | reativo |
| PIS/COFINS | PIS/COFINS de cada linha, repartido entre E e D pela receita (TA) de cada unidade |
| ICMS | sem valor (a RNF não traz ICMS) |

Itens da base econômica sem grupo no modelo ficam fora do layout e aparecem na conferência, como no
Faturamento. A base, em R$, é gravada no relatório detalhado (aba `Layout_base`): é dela que o mês
seguinte lê o estorno.

In [ ]:
import importlib.util

# Receita Faturada converter of the Faturamento: model, header and item -> layout group de-para
os.environ["PXQ_BASE"] = str(BASE.parent / "BW")
_spec = importlib.util.spec_from_file_location(
    "gera_receita_faturada", BASE.parent / "BW" / "3-Py" / "1-Entrega" / "gera_receita_faturada.py")
conv = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(conv)
_, CAB, DE_PARA, GRADE = conv.le_modelo()
COL_UDER, COL_OUTROS = conv.COL_UDER, conv.COL_OUTROS
VALORES_LAY = [c for c in CAB.values() if c not in conv.COL_ID]
VALORES_LAY[VALORES_LAY.index(CAB["PIS/COFINS"]):VALORES_LAY.index(CAB["PIS/COFINS"])] = [COL_UDER, COL_OUTROS]
COLS_MONET_LAY = [c for c in VALORES_LAY if c not in (CAB["CONSUMO"], CAB["TOTAL"])]
CHAVE_LAY = ["TipoFaturamento", "OpcaoCompra", "Classe", "Unidade"]

TIPO_RNF = {"SAP ZUBIL_146": "RNF", "SAP ZUBIL_146 (demanda livre)": "RNF", "TOI (CNR)": "RNF_TOI",
            "Faturas pendentes": "RNF_PENDENTES", "Art. 323": "RNF_ART323", "Ajuste de carga": "RNF_CARGA",
            "Iluminação pública": "RNF_IP", "Ajuste livre": "RNF_AJUSTE_LIVRE",
            "Bandeira dos ajustes": "RNF_BANDEIRA"}
TIPOS_RNF = ["RNF", "RNF_TOI", "RNF_PENDENTES", "RNF_ART323", "RNF_CARGA", "RNF_IP", "RNF_AJUSTE_LIVRE",
             "RNF_BANDEIRA"]
CLASSE_LAY = {"RESIDENCIAL": "Residencial", "INDUSTRIAL": "Industrial", "COMERCIAL": "Comercial",
              "RURAL": "Rural", "PODER PÚBLICO": "Poderes Públicos", "ILUMINAÇÃO PÚBLICA": "Iluminação Pública",
              "SERVIÇO PÚBLICO": "Serviços Públicos", "CONSUMO PRÓPRIO": "Consumo Próprio"}

def classe_lay(classe_rel, beneficio=None, baixa_renda=False):
    """Layout class; social benefits (Tarifa Social, Desconto Social) go to Residencial Baixa Renda."""
    if classe_rel == "RESIDENCIAL" and (baixa_renda or (isinstance(beneficio, str) and beneficio)):
        return "Residencial Baixa Renda"
    return CLASSE_LAY.get(classe_rel)

grupo_item = {c: DE_PARA.get(c) for c in COMPONENTES}
reg, fora = [], []
for i, r in linhas.iterrows():
    tipo, cl = TIPO_RNF[r["ORIGEM"]], classe_lay(r["CLASSE_REL"], r["BENEFICIO"])
    ta_u = {u: B2U[u]["TA"].loc[i].sum() for u in UNIDADES}
    ta_tot = sum(ta_u.values())
    for u in UNIDADES:
        reg_u = {"TipoFaturamento": tipo, "OpcaoCompra": r["MERCADO"], "Classe": cl, "Unidade": u,
                 CAB["CONSUMO"]: r["MWH"] if u == "E" else r["KW"] / 1000}
        sem_grupo = 0.0
        for c, v in B2U[u]["BE"].loc[i].items():
            if grupo_item[c]:
                reg_u[grupo_item[c]] = reg_u.get(grupo_item[c], 0.0) + v
            else:
                sem_grupo += v
        reg_u[CAB["BASE FINANCEIRA"]] = B2U[u]["FIN"].loc[i].sum()
        reg_u[CAB["BASE CVA"]] = B2U[u]["CVA"].loc[i].sum()
        parte = ta_u[u] / ta_tot if ta_tot else (1.0 if u == "E" else 0.0)
        reg_u[CAB["PIS/COFINS"]] = r["PIS_COFINS"] * parte
        reg_u[COL_OUTROS] = r["RESIDUO"] if u == "E" else 0.0
        reg.append(reg_u)
        if abs(sem_grupo) > 0.005:
            fora.append(("Item da base econômica sem grupo no modelo", f"{tipo} / {r['MERCADO']} / {cl} / {u}", sem_grupo))
# non-tariff: reactive and flag of the ZUBIL, flag of the manual adjustments
for (merc, cl, ben), v in sap.groupby(["MERCADO", "CLASSE_REL", "BENEFICIO"], dropna=False)[["R_REATIVO", "R_BANDEIRA"]].sum().iterrows():
    reg.append({"TipoFaturamento": "RNF", "OpcaoCompra": merc, "Classe": classe_lay(cl, ben), "Unidade": "E",
                COL_UDER: v["R_REATIVO"], CAB["BANDEIRA"]: v["R_BANDEIRA"]})
for _, r in man[man["BANDEIRA"] != 0].iterrows():
    cb = str(r["CLASSE_BASE"])
    reg.append({"TipoFaturamento": TIPO_RNF[r["ORIGEM"]], "OpcaoCompra": r["MERCADO"],
                "Classe": classe_lay(CLASSE_BASE.get(cb), baixa_renda="BAIXA RENDA" in cb), "Unidade": "E",
                CAB["BANDEIRA"]: r["BANDEIRA"]})

lay = pd.DataFrame(reg)
lay["Classe"] = lay["Classe"].fillna("(sem classe)")
lay = lay.groupby(CHAVE_LAY, sort=False).sum(numeric_only=True).reindex(columns=VALORES_LAY).fillna(0.0)
lay[CAB["TOTAL"]] = lay[[c for c in VALORES_LAY[VALORES_LAY.index(CAB["PARCELA B"]):VALORES_LAY.index(CAB["BASE CVA"]) + 1]]].sum(axis=1)
na_grade = {(op, cl, u) for _, op, cl, u in GRADE}
fora_grade = [k for k in lay.index if k[1:] not in na_grade and lay.loc[k, COLS_MONET_LAY].abs().sum() > 0.005]
for k in fora_grade:
    fora.append(("Opção x classe x unidade fora da grade do modelo", " / ".join(k), lay.loc[k, COLS_MONET_LAY].sum()))
lay = lay.drop(index=fora_grade)
lay = lay[(lay[COLS_MONET_LAY + [CAB["CONSUMO"]]].abs() > 1e-9).any(axis=1)]
LAYOUT_BASE = lay.reset_index()
FORA_LAY = pd.DataFrame(fora, columns=["Categoria", "Detalhe", "R$"])

total_lay = LAYOUT_BASE[COLS_MONET_LAY].sum().sum()
total_rnf = CONCILIACAO.loc["Total", "MR$"] * M
print(f"layout: {len(LAYOUT_BASE)} linhas | gravado MR$ {total_lay / M:,.2f} | fora MR$ {FORA_LAY['R$'].sum() / M:,.2f} "
      f"| total RNF MR$ {total_rnf / M:,.2f} | diferenca MR$ {(total_rnf - total_lay - FORA_LAY['R$'].sum()) / M:,.4f}")
assert abs(total_rnf - total_lay - FORA_LAY["R$"].sum()) < 1.0, "layout nao fecha com o total da RNF"
print(LAYOUT_BASE.groupby("TipoFaturamento", sort=False)[COLS_MONET_LAY].sum().sum(axis=1).div(M)
      .to_string(float_format="{:,.2f}".format))

---
## Passo 13 — Gravar o relatório detalhado

`1-Entrega/rnf/rnf_<REFERENCIA>.xlsx`. Valores em MR$, energia em MWh e demanda em MW.

A RNF é analisada pelo efeito na receita: a provisão do mês menos o estorno da provisão do mês
anterior. No acumulado do ano, os estornos intermediários se anulam e o efeito é a provisão do mês
menos a provisão de dezembro do ano anterior. As quatro primeiras abas trazem essa visão, no mesmo
recorte (origem, mercado, classe, subgrupo e modalidade), com os 37 itens e as sete famílias na
tarifa de aplicação; reativo, bandeira e PIS/COFINS entram como linhas não tarifáveis, para que o
total feche com a RNF contabilizada.

| Aba | Conteúdo |
|---|---|
| `Provisao` | provisão do mês |
| `Estorno` | estorno da provisão do mês anterior (lida da saída já gravada daquele mês, com sinal trocado) |
| `Saldo` | provisão + estorno: efeito da RNF na receita do mês |
| `Saldo_YTD` | provisão do mês − provisão de dezembro do ano anterior: efeito acumulado no ano (só quando a saída de dezembro existe) |
| `Layout_base` | a provisão no layout Receita Faturada, em R$ (Passo 12); fonte do estorno do mês seguinte |
| `Componentes` | uma linha por origem, classe, subclasse e categoria: bloco 1 e bloco 2 na tarifa de aplicação, 37 itens |
| `Componentes_BE` | bloco 2 na base econômica |
| `Familias` | bloco 2 nas sete famílias e quatro camadas |
| `Resumo` | MWh, valor (MR$) e tarifa média (R$/MWh) por origem, mercado e classe |
| `Nao_tarifaveis` | PIS/COFINS, reativo e bandeira por mercado e classe |
| `Conciliacao` | total aberto + não tarifáveis × total da `Base RNF` |
| `Conciliacao_Base_RNF` | comparação bloco a bloco com a `Base RNF` |

In [ ]:
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

IDENT = ["ORIGEM", "MERCADO", "CLASSE_REL", "CL1", "SUB_SAP", "CATEGORIA", "SUBGRUPO", "MODALIDADE",
         "SUBCLASSE", "BENEFICIO", "POSTO_E", "POSTO_D", "MWH", "MW", "VALOR_BRUTO", "PIS_COFINS", "VALOR_LIQ",
         "BLOCO1_TA", "DIFERENCA", "REGIME", "DESCONTO", "AJUSTE_FIO", "AJUSTE_FATURADO", "FATOR", "FAIXA2_PCT",
         "RESIDUO", "BLOCO2_TA"]
MONETARIOS = ["VALOR_BRUTO", "PIS_COFINS", "VALOR_LIQ", "BLOCO1_TA", "DIFERENCA", "DESCONTO", "AJUSTE_FIO",
              "AJUSTE_FATURADO", "RESIDUO", "BLOCO2_TA"]
ROTULO = {"ORIGEM": "Origem", "MERCADO": "Mercado", "CLASSE_REL": "Classe", "CL1": "Classe de origem",
          "SUB_SAP": "Subclasse SAP", "CATEGORIA": "Categoria tarifa", "SUBGRUPO": "Subgrupo",
          "MODALIDADE": "Modalidade", "SUBCLASSE": "Subclasse (tarifa)", "BENEFICIO": "Benefício social",
          "POSTO_E": "Posto da energia", "POSTO_D": "Posto da demanda",
          "MWH": "RNF (MWh)", "MW": "Demanda (MW)", "VALOR_BRUTO": "Valor com PIS/COFINS (MR$)",
          "PIS_COFINS": "PIS/COFINS (MR$)", "VALOR_LIQ": "Valor líquido, alvo do bloco 2 (MR$)",
          "BLOCO1_TA": "Bloco 1 TA (MR$)", "DIFERENCA": "Diferença (MR$)", "REGIME": "Regime do bloco 2",
          "DESCONTO": "Aplicação do desconto no fio (MR$)", "AJUSTE_FIO": "Ajuste no fio (MR$)",
          "AJUSTE_FATURADO": "Postos não valorizados, pela composição do faturado (MR$)",
          "FATOR": "Fator proporcional", "FAIXA2_PCT": "% energia na faixa 02 (estimada)",
          "RESIDUO": "Resíduo (MR$)", "BLOCO2_TA": "Bloco 2 TA (MR$)"}
saida = linhas.copy()
saida["MW"] = saida["KW"] / 1000
saida[MONETARIOS] = saida[MONETARIOS] / M
esq = saida[IDENT].rename(columns=ROTULO)
esq.insert(0, "Referência", REFERENCIA)
esq.insert(1, "Vigência tarifária", ANO_VIGENCIA)

# tariff items in MR$
aba_comp = pd.concat([esq, (B1["TA"] / M).add_prefix("[1] "), (B2["TA"] / M).add_prefix("[2] ")], axis=1)
aba_be = pd.concat([esq, (B2["BE"] / M).add_prefix("[2] BE · ")], axis=1)
fam = pd.DataFrame({f"[2] {c} · {f}": B2[c][l].sum(axis=1) / M for c in CAMADAS for f, l in FAMILIA.items()})
aba_fam = pd.concat([esq, fam], axis=1)
aba_res = (linhas.groupby(["ORIGEM", "MERCADO", "CLASSE_REL"])[["MWH", "VALOR_BRUTO", "VALOR_LIQ", "BLOCO1_TA"]]
           .sum().reset_index())
aba_res["R$/MWh"] = (aba_res["VALOR_LIQ"] / aba_res["MWH"]).replace([np.inf, -np.inf], np.nan)
aba_res[["VALOR_BRUTO", "VALOR_LIQ", "BLOCO1_TA"]] /= M
aba_res = aba_res.rename(columns=ROTULO)

nt_sap = sap.groupby(["MERCADO", "CLASSE_REL"])[["R_REATIVO", "R_BANDEIRA"]].sum()
nt = pd.concat([nt_sap["R_REATIVO"].rename("Reativo (MR$)"),
                nt_sap["R_BANDEIRA"].add(BANDEIRA_AJUSTES, fill_value=0.0).rename("Bandeira (MR$)"),
                linhas.groupby(["MERCADO", "CLASSE_REL"])["PIS_COFINS"].sum().rename("PIS/COFINS (MR$)")],
               axis=1).fillna(0.0) / M
aba_nt = nt.reset_index().rename(columns={"MERCADO": "Mercado", "CLASSE_REL": "Classe"})

# ---------------------------------------------------------------- provision, reversal and balance
CHAVE_SALDO = ["Origem", "Mercado", "Classe", "Subgrupo", "Modalidade"]
VAL_TOTAL = "Total RNF (MR$)"
fam_ta = pd.DataFrame({f"Família · {f}": B2["TA"][l].sum(axis=1) for f, l in FAMILIA.items()})
base_prov = pd.concat([linhas[["ORIGEM", "MERCADO", "CLASSE_REL", "SUBGRUPO", "MODALIDADE", "MWH", "KW"]],
                       B2["TA"].add_prefix("Item · "), fam_ta], axis=1)
base_prov["Bloco 2 TA"] = linhas["BLOCO2_TA"] + linhas["RESIDUO"]
base_prov.columns = CHAVE_SALDO + ["Energia (MWh)", "Demanda (MW)"] + list(base_prov.columns[7:])
num_cols = [c for c in base_prov.columns if c not in CHAVE_SALDO]
prov = base_prov.groupby(CHAVE_SALDO, as_index=False)[num_cols].sum()
prov["Demanda (MW)"] /= 1000
monet = [c for c in num_cols if c not in ("Energia (MWh)", "Demanda (MW)")]
prov[monet] /= M
prov = prov.rename(columns={c: f"{c} (MR$)" for c in monet})
prov[VAL_TOTAL] = prov["Bloco 2 TA (MR$)"]
# non-tariff rows: reactive, flag and PIS/COFINS by market and class
for rotulo, col in (("Reativo", "Reativo (MR$)"), ("Bandeira", "Bandeira (MR$)"), ("PIS/COFINS", "PIS/COFINS (MR$)")):
    x = aba_nt[["Mercado", "Classe", col]].rename(columns={col: VAL_TOTAL})
    x = x[x[VAL_TOTAL] != 0].assign(Origem=f"Não tarifável · {rotulo}", Subgrupo="", Modalidade="")
    prov = pd.concat([prov, x], ignore_index=True)
prov = prov.fillna({c: 0.0 for c in prov.columns if c not in CHAVE_SALDO})
assert abs(prov[VAL_TOTAL].sum() - CONCILIACAO.loc["Total", "MR$"]) < 1e-6, "provisao nao fecha com o total"

def le_provisao(ref):
    """Provision sheet of an output already written for another month, or None."""
    arq = DIR_SAIDA / f"rnf_{ref}.xlsx"
    if not arq.exists():
        return None
    x = pd.read_excel(arq, sheet_name="Provisao")
    x[CHAVE_SALDO] = x[CHAVE_SALDO].fillna("")
    return x

def soma(*dfs):
    """Sum frames on the balance key (outer join)."""
    x = pd.concat(dfs, ignore_index=True)
    cols = [c for c in x.columns if c not in CHAVE_SALDO]
    return x.groupby(CHAVE_SALDO, as_index=False, sort=False)[cols].sum()

def sinal_trocado(df):
    x = df.copy()
    cols = [c for c in x.columns if c not in CHAVE_SALDO]
    x[cols] = -x[cols]
    return x

def sem_dados(msg):
    return pd.DataFrame({"Aviso": [msg]})

prov[CHAVE_SALDO] = prov[CHAVE_SALDO].fillna("")
REF_ANT = (ALVO - pd.offsets.MonthBegin(1)).strftime("%Y-%m")
REF_DEZ = f"{int(AAAA) - 1}-12"
ant, dez = le_provisao(REF_ANT), le_provisao(REF_DEZ)
if ant is not None:
    estorno = sinal_trocado(ant)
    saldo = soma(prov, estorno)
    print(f"estorno de {REF_ANT}: MR$ {estorno[VAL_TOTAL].sum():,.2f} | saldo do mes: MR$ {saldo[VAL_TOTAL].sum():,.2f}")
else:
    estorno = sem_dados(f"Provisão de {REF_ANT} não encontrada em {DIR_SAIDA.name}: rode o mês anterior primeiro.")
    saldo = sem_dados(f"Saldo indisponível: falta a provisão de {REF_ANT}.")
    print(f"ATENCAO: sem a saida de {REF_ANT}; abas Estorno e Saldo sem dados")
if dez is not None:
    saldo_ytd = soma(prov, sinal_trocado(dez))
    print(f"saldo YTD (vs {REF_DEZ}): MR$ {saldo_ytd[VAL_TOTAL].sum():,.2f}")
else:
    saldo_ytd = None
    print(f"sem a saida de {REF_DEZ}: aba Saldo_YTD nao gravada")
print(f"provisao do mes: MR$ {prov[VAL_TOTAL].sum():,.2f}")

def formata(ws, n_ident):
    ws.freeze_panes = ws.cell(row=2, column=n_ident + 1)
    for cel in ws[1]:
        cel.font = Font(bold=True, size=9, color="FFFFFF")
        cel.fill = PatternFill("solid", fgColor="1F4E79")
        cel.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
    ws.row_dimensions[1].height = 42
    for col in range(1, ws.max_column + 1):
        titulo = str(ws.cell(row=1, column=col).value)
        ws.column_dimensions[get_column_letter(col)].width = 18 if col <= n_ident else 15
        fmt = "0.0%" if "%" in titulo else ("0.0000" if ("Fator" in titulo or "MR$" in titulo) else "#,##0.00")
        for lin in range(2, ws.max_row + 1):
            if isinstance(ws.cell(row=lin, column=col).value, (int, float)):
                ws.cell(row=lin, column=col).number_format = fmt
    ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"

DIR_SAIDA.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(SAIDA, engine="openpyxl") as w:
    n = esq.shape[1]
    abas_saldo = [("Provisao", prov), ("Estorno", estorno), ("Saldo", saldo)]
    if saldo_ytd is not None:
        abas_saldo.append(("Saldo_YTD", saldo_ytd))
    for nome, df in abas_saldo:
        df.to_excel(w, sheet_name=nome, index=False)
        formata(w.book[nome], len(CHAVE_SALDO) if "Aviso" not in df.columns else 1)
    for nome, df, ni in (("Componentes", aba_comp, n), ("Componentes_BE", aba_be, n),
                         ("Familias", aba_fam, n), ("Resumo", aba_res, 3), ("Nao_tarifaveis", aba_nt, 2),
                         ("Conciliacao", CONCILIACAO.reset_index(names="Item"), 1),
                         ("Layout_base", LAYOUT_BASE, len(CHAVE_LAY)),
                         ("Conciliacao_Base_RNF", CONC_BASE_RNF.reset_index(names="Bloco"), 1)):
        df.to_excel(w, sheet_name=nome, index=False)
        formata(w.book[nome], ni)
print(f"gravado: {SAIDA}")

---
## Passo 14 — Gravar o consolidado (layout Receita Não Faturada)

`1-Entrega/receita_nao_faturada/receita_nao_faturada_<REFERENCIA>.xlsx`, a partir do modelo da área,
no mesmo layout e na mesma unidade da `receita_faturada` (R$ mil; quantidade em MWh ou MW):

| Aba | Conteúdo |
|---|---|
| `Provisao` | provisão do mês |
| `Estorno` | estorno da provisão do mês anterior (aba `Layout_base` do relatório detalhado daquele mês, com sinal trocado) |
| `Saldo` | provisão + estorno: efeito da RNF na receita do mês |
| `Saldo_YTD` | provisão − provisão de dezembro do ano anterior (só quando a saída de dezembro existe) |
| `Conferencia` | total gravado + fora do layout = total da RNF |

In [ ]:
import copy
import datetime as dt

DIR_CONSOLIDADO = BASE / "1-Entrega" / "receita_nao_faturada"
SAIDA_CONSOLIDADO = DIR_CONSOLIDADO / f"receita_nao_faturada_{REFERENCIA}.xlsx"

def le_layout(ref):
    """Layout_base of the detailed output of another month (R$), indexed by the layout key, or None."""
    arq = DIR_SAIDA / f"rnf_{ref}.xlsx"
    if not arq.exists():
        return None
    try:
        x = pd.read_excel(arq, sheet_name="Layout_base")
    except ValueError:
        return None
    return x.set_index(CHAVE_LAY).reindex(columns=VALORES_LAY).fillna(0.0)

prov_lay = LAYOUT_BASE.set_index(CHAVE_LAY)[VALORES_LAY]
ant_lay, dez_lay = le_layout(REF_ANT), le_layout(REF_DEZ)
abas_lay = {"Provisao": prov_lay}
if ant_lay is not None:
    abas_lay["Estorno"] = -ant_lay
    abas_lay["Saldo"] = prov_lay.add(-ant_lay, fill_value=0.0)
else:
    abas_lay["Estorno"] = abas_lay["Saldo"] = None
if dez_lay is not None:
    abas_lay["Saldo_YTD"] = prov_lay.add(-dez_lay, fill_value=0.0)

# template sheet: model header with the UD/ER and Outros columns before PIS/COFINS (as in the Faturamento)
wb = openpyxl.load_workbook(conv.ARQ_MODELO)
modelo = wb["Receita Faturada"]
larguras = {c: modelo.column_dimensions[get_column_letter(c)].width for c in range(1, modelo.max_column + 1)}
c_pis = next(c for c in range(2, modelo.max_column + 1) if modelo.cell(3, c).value == CAB["PIS/COFINS"])
modelo.insert_cols(c_pis, 2)
for k, nome in enumerate([COL_UDER, COL_OUTROS]):
    for lin in (2, 3):
        modelo.cell(lin, c_pis + k).value = modelo.cell(lin, c_pis + 2).value if lin == 2 else nome
        modelo.cell(lin, c_pis + k)._style = copy.copy(modelo.cell(lin, c_pis + 2)._style)
for c, w in larguras.items():
    if w:
        modelo.column_dimensions[get_column_letter(c if c < c_pis else c + 2)].width = w
for c in (c_pis, c_pis + 1):
    modelo.column_dimensions[get_column_letter(c)].width = 10
for lin in (1, 2):
    for c in range(1, modelo.max_column + 1):
        v = modelo.cell(lin, c).value
        if isinstance(v, str) and "FATURAD" in v.upper():
            modelo.cell(lin, c).value = v.upper().replace("RECEITA FATURADA", "RECEITA NÃO FATURADA")
estilos = [copy.copy(modelo.cell(4, c)._style) for c in range(1, modelo.max_column + 1)]
modelo.delete_rows(4, modelo.max_row)
col_de = {modelo.cell(3, c).value: c for c in range(2, modelo.max_column + 1)}
letra_i, letra_z = (get_column_letter(col_de[CAB[k]]) for k in ("PARCELA B", "BASE CVA"))
competencia = dt.datetime(int(AAAA), int(MM), 1)

def preenche(ws, tabela, aviso=None):
    """Write the layout rows of `tabela` (R$) in R$ thousand, in the model grid order."""
    if tabela is None:
        ws.cell(4, 2, aviso)
        return 0.0
    lin, total = 4, 0.0
    for tipo in TIPOS_RNF:
        for _, op, cl, u in GRADE:
            k = (tipo, op, cl, u)
            if k not in tabela.index:
                continue
            r = tabela.loc[k]
            if r[COLS_MONET_LAY].abs().sum() < 0.005 and abs(r[CAB["CONSUMO"]]) < 1e-9:
                continue
            for nome, v in zip(conv.COL_ID, ["Receita Não Faturada", competencia, tipo, op, cl, u]):
                ws.cell(lin, col_de[nome], v)
            for nome in VALORES_LAY:
                if nome == CAB["TOTAL"]:
                    continue
                v = r[nome] if nome == CAB["CONSUMO"] else r[nome] / 1000          # R$ -> R$ mil
                ws.cell(lin, col_de[nome], float(v))
            ws.cell(lin, col_de[CAB["TOTAL"]], f"=SUM({letra_i}{lin}:{letra_z}{lin})")
            for c, st in enumerate(estilos, start=1):
                ws.cell(lin, c)._style = copy.copy(st)
            ws.cell(lin, col_de["Competência"]).number_format = "dd/mm/yyyy"
            total += r[COLS_MONET_LAY].sum()
            lin += 1
    return total

gravado = {}
for nome, tab in abas_lay.items():
    ws = wb.copy_worksheet(modelo)
    ws.title = nome
    aviso = f"{nome} indisponível: falta o relatório detalhado de {REF_ANT} (rnf_{REF_ANT}.xlsx)."
    gravado[nome] = preenche(ws, tab, aviso)
wb.remove(modelo)

ws_c = wb.create_sheet("Conferencia")
ws_c.append(["Conciliação da provisão (R$)", None])
for item, v in (("Total da RNF (aberto + PIS/COFINS + reativo + bandeira)", total_rnf),
                ("Gravado no layout (Parcela B a ICMS)", gravado["Provisao"]),
                ("Fora do layout (detalhe abaixo)", FORA_LAY["R$"].sum()),
                ("Diferença (deve ser zero)", total_rnf - gravado["Provisao"] - FORA_LAY["R$"].sum())):
    ws_c.append([item, v])
ws_c.append([])
ws_c.append(["Categoria fora do layout", "Detalhe", "R$"])
for r in FORA_LAY.itertuples(index=False):
    ws_c.append(list(r))
ws_c.append([])
ws_c.append(["Gravado por aba (R$)", None])
for nome, v in gravado.items():
    ws_c.append([nome, v])
for row in ws_c.iter_rows():
    for cel in row:
        if isinstance(cel.value, float):
            cel.number_format = "#,##0.00"
ws_c.column_dimensions["A"].width = 62
ws_c.column_dimensions["B"].width = 48
ws_c.column_dimensions["C"].width = 18

DIR_CONSOLIDADO.mkdir(parents=True, exist_ok=True)
wb.save(SAIDA_CONSOLIDADO)
print(" | ".join(f"{k} MR$ {v / M:,.2f}" for k, v in gravado.items()))
print(f"gravado: {SAIDA_CONSOLIDADO}")